# MedStock-Wise: Healthcare Inventory Demand Prediction

A predictive inventory optimization framework combining LightGBM demand forecasting with EOQ-based replenishment recommendations.

## 1. Setup & Synthetic Dataset Generation

Generate 2,500 inventory records (5 items × 500 days) with realistic demand variability: weekday/weekend cycles, seasonal surges (Nov–Feb), long-term growth trend (15%), and 8% item-level noise.

In [51]:
%pip install -q seaborn scikit-learn lightgbm



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [52]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="pastel")

# ── Reproducibility ──────────────────────────────────────────────────────────
np.random.seed(42)

# ── Inventory dataset: 5 items × 500 days ────────────────────────────────────
dates = pd.date_range('2024-10-01', '2026-02-12', freq='D')
items = {
    'Ventilator'   : {'base': 45,  'vendor': 'V001', 'type': 'Equipment',  'cost': 85000},
    'Surgical Mask': {'base': 420, 'vendor': 'V001', 'type': 'Consumable', 'cost': 12},
    'IV Drip'      : {'base': 280, 'vendor': 'V002', 'type': 'Consumable', 'cost': 85},
    'Gloves'       : {'base': 500, 'vendor': 'V002', 'type': 'Consumable', 'cost': 8},
    'X-ray Machine': {'base': 18,  'vendor': 'V003', 'type': 'Equipment',  'cost': 120000},
}

records = []
for item_name, cfg in items.items():
    base = cfg['base']
    stock = base * 50

    for i, date in enumerate(dates):
        day_of_week = date.dayofweek
        month = date.month

        weekly_effect = 1.15 if day_of_week < 5 else 0.82
        seasonal = 1.25 if month in [1, 2, 11, 12] else 1.0
        trend = 1 + (i / len(dates)) * 0.18

        # Non-linear demand dynamics create realistic regime changes.
        flu_wave = 1.35 if (date.month in [12, 1, 2] and item_name in ['Surgical Mask', 'Gloves']) else 1.0
        emergency_spike = 1.25 if (i % 45 in range(4) and item_name in ['Ventilator', 'IV Drip']) else 1.0
        interaction_boost = 1.0 + 0.18 * (day_of_week <= 1) * (month in [1, 11, 12])

        noise = np.random.normal(1.0, 0.07)
        usage = max(1, int(base * weekly_effect * seasonal * trend * flu_wave * emergency_spike * interaction_boost * noise))

        # Restock logic introduces both shortage and non-shortage periods.
        reorder_point = int(base * 8.5)
        restock_qty = int(base * np.random.uniform(10, 18)) if stock < reorder_point else 0
        stock = min(int(base * 70), max(0, stock - usage + restock_qty))

        records.append({
            'Date': date.strftime('%Y-%m-%d'),
            'Item_ID': 100 + list(items.keys()).index(item_name),
            'Item_Type': cfg['type'],
            'Item_Name': item_name,
            'Current_Stock': int(stock),
            'Min_Required': int(base * 6),
            'Max_Capacity': int(base * 70),
            'Unit_Cost': cfg['cost'] * np.round(np.random.uniform(0.9, 1.1), 2),
            'Avg_Usage_Per_Day': usage,
            'Restock_Lead_Time': np.random.randint(3, 21),
            'Vendor_ID': cfg['vendor'],
        })

inventory_df = pd.DataFrame(records)
print(f"Generated {len(inventory_df)} inventory records")
print(inventory_df.groupby('Item_Name')['Avg_Usage_Per_Day'].describe().round(1))

# ── Supplementary datasets (load from same directory) ────────────────────────
import os

sample_data_dir = '/content/sample_data/'

patient_df = pd.read_csv('patient_data.csv')
staff_df = pd.read_csv('staff_data.csv')
financial_df = pd.read_csv('financial_data.csv')
vendor_df = pd.read_csv('vendor_data.csv')

print(f"\nSupplementary datasets loaded:")
print(f"  Patient  : {patient_df.shape}")
print(f"  Staff    : {staff_df.shape}")
print(f"  Financial: {financial_df.shape}")
print(f"  Vendor   : {vendor_df.shape}")


Generated 2500 inventory records
               count   mean    std    min    25%    50%    75%     max
Item_Name                                                             
Gloves         500.0  737.5  247.0  359.0  579.0  656.0  884.5  1475.0
IV Drip        500.0  375.9   92.1  203.0  320.0  359.0  433.0   688.0
Surgical Mask  500.0  620.2  208.4  293.0  486.0  554.0  760.2  1203.0
Ventilator     500.0   59.7   14.1   33.0   50.0   58.0   68.2   103.0
X-ray Machine  500.0   23.1    5.4   13.0   20.0   22.5   26.2    42.0

Supplementary datasets loaded:
  Patient  : (500, 10)
  Staff    : (500, 9)
  Financial: (500, 4)
  Vendor   : (3, 7)


## 2. Data Quality Check

Summarise each dataset for column types, null values, and duplicates.

In [53]:
def check(df):
    """Returns a concise quality summary of a DataFrame."""
    summary = [
        [col, df[col].dtype, df[col].count(), df[col].nunique(),
         df[col].isnull().sum(), df.duplicated().sum()]
        for col in df.columns
    ]
    return pd.DataFrame(summary,
                        columns=["column", "dtype", "instances",
                                 "unique", "sum_null", "duplicates"])

for name, df in [("Inventory", inventory_df), ("Patient", patient_df),
                  ("Staff", staff_df), ("Financial", financial_df),
                  ("Vendor", vendor_df)]:
    print(f"\n{'='*10} {name} Dataset {'='*10}")
    display(check(df))
    display(df.head(2))



========== Inventory Dataset ==========


,column,dtype,instances,unique,sum_null,duplicates
0,Date,str,2500,500,0,0
1,Item_ID,int64,2500,5,0,0
2,Item_Type,str,2500,2,0,0
3,Item_Name,str,2500,5,0,0
4,Current_Stock,int64,2500,1949,0,0
5,Min_Required,int64,2500,5,0,0
6,Max_Capacity,int64,2500,5,0,0
7,Unit_Cost,float64,2500,105,0,0
8,Avg_Usage_Per_Day,int64,2500,779,0,0
9,Restock_Lead_Time,int64,2500,18,0,0


,Date,Item_ID,Item_Type,Item_Name,Current_Stock,Min_Required,Max_Capacity,Unit_Cost,Avg_Usage_Per_Day,Restock_Lead_Time,Vendor_ID
0,2024-10-01,100,Equipment,Ventilator,2184,270,3150,89250.0,66,9,V001
1,2024-10-02,100,Equipment,Ventilator,2120,270,3150,84150.0,64,13,V001



========== Patient Dataset ==========


,column,dtype,instances,unique,sum_null,duplicates
0,Patient_ID,str,500,500,0,0
1,Admission_Date,str,500,500,0,0
2,Discharge_Date,str,500,500,0,0
3,Primary_Diagnosis,str,500,4,0,0
4,Procedure_Performed,str,500,4,0,0
5,Room_Type,str,500,2,0,0
6,Bed_Days,int64,500,14,0,0
7,Supplies_Used,str,500,3,0,0
8,Equipment_Used,str,500,3,0,0
9,Staff_Needed,str,500,3,0,0


...

,column,dtype,instances,unique,sum_null,duplicates
0,Date,str,500,500,0,0
1,Expense_Category,str,500,3,0,0
2,Amount,float64,500,500,0,0
3,Description,str,500,3,0,0


,Date,Expense_Category,Amount,Description
0,2024-10-01,Staffing,29391.86,Surgical masks
1,2024-10-02,Supplies,47757.71,Surgical masks



========== Vendor Dataset ==========


,column,dtype,instances,unique,sum_null,duplicates
0,Vendor_ID,str,3,3,0,0
1,Vendor_Name,str,3,3,0,0
2,Item_Supplied,str,3,3,0,0
3,Avg_Lead_Time (days),int64,3,3,0,0
4,Cost_Per_Item,float64,3,3,0,0
5,Last_Order_Date,str,3,3,0,0
6,Next_Delivery_Date,str,3,3,0,0


,Vendor_ID,Vendor_Name,Item_Supplied,Avg_Lead_Time (days),Cost_Per_Item,Last_Order_Date,Next_Delivery_Date
0,V001,MedSupplies Inc.,Surgical Mask,5,0.5,2024-09-28,2024-10-03
1,V002,EquipMed Co.,Ventilator,30,20000.0,2024-09-01,2024-10-15


## 3. Data Preprocessing

Parse dates, align supplementary datasets to the inventory timeline, and merge into a single combined DataFrame.

In [54]:
# ── Parse date columns ────────────────────────────────────────────────────────
inventory_df = inventory_df.rename(columns={'Date': 'Inventory_Date'})
financial_df = financial_df.rename(columns={'Date': 'Financial_Date'})

inventory_df['Inventory_Date'] = pd.to_datetime(inventory_df['Inventory_Date'])
patient_df['Admission_Date'] = pd.to_datetime(patient_df['Admission_Date'])
patient_df['Discharge_Date'] = pd.to_datetime(patient_df['Discharge_Date'])
staff_df['Shift_Date'] = pd.to_datetime(staff_df['Shift_Date'])
financial_df['Financial_Date'] = pd.to_datetime(financial_df['Financial_Date'])

# ── Align supplementary records to inventory length ───────────────────────────
patient_df = patient_df.iloc[:len(inventory_df)].reset_index(drop=True)
staff_df = staff_df.iloc[:len(inventory_df)].reset_index(drop=True)
financial_df = financial_df.iloc[:len(inventory_df)].reset_index(drop=True)

# ── Merge into combined DataFrame ─────────────────────────────────────────────
combined_df = pd.concat(
    [inventory_df.reset_index(drop=True), patient_df, staff_df, financial_df],
    axis=1
)

# Fill missing values with dtype-aware defaults to avoid pandas dtype errors.
num_cols = combined_df.select_dtypes(include=[np.number]).columns
str_cols = combined_df.select_dtypes(include=['object', 'string', 'category']).columns
datetime_cols = combined_df.select_dtypes(include=['datetime64[ns]', 'datetimetz']).columns

combined_df[num_cols] = combined_df[num_cols].fillna(0)
combined_df[str_cols] = combined_df[str_cols].fillna('')

# For datetimes, keep timestamp dtype and fill from nearby values.
if len(datetime_cols) > 0:
    combined_df[datetime_cols] = combined_df[datetime_cols].ffill().bfill()

# ── CRITICAL FIX: sort by date so temporal split covers ALL items ──────────────
combined_df = combined_df.sort_values('Inventory_Date').reset_index(drop=True)

print(f"Combined dataset shape: {combined_df.shape}")
print(f"Columns: {list(combined_df.columns)}")


Combined dataset shape: (2500, 34)
Columns: ['Inventory_Date', 'Item_ID', 'Item_Type', 'Item_Name', 'Current_Stock', 'Min_Required', 'Max_Capacity', 'Unit_Cost', 'Avg_Usage_Per_Day', 'Restock_Lead_Time', 'Vendor_ID', 'Patient_ID', 'Admission_Date', 'Discharge_Date', 'Primary_Diagnosis', 'Procedure_Performed', 'Room_Type', 'Bed_Days', 'Supplies_Used', 'Equipment_Used', 'Staff_Needed', 'Staff_ID', 'Staff_Type', 'Shift_Date', 'Shift_Start_Time', 'Shift_End_Time', 'Current_Assignment', 'Hours_Worked', 'Patients_Assigned', 'Overtime_Hours', 'Financial_Date', 'Expense_Category', 'Amount', 'Description']


## 4. Lag Feature Engineering

Create 1-day, 3-day, 7-day lag features and a 7-day rolling mean of prior demand per item. This gives the model temporal context for each prediction.

In [55]:
inventory_df = inventory_df.sort_values(['Item_Name', 'Inventory_Date']).copy()

inventory_df['Usage_Lag_1']      = inventory_df.groupby('Item_Name')['Avg_Usage_Per_Day'].shift(1)
inventory_df['Usage_Lag_3']      = inventory_df.groupby('Item_Name')['Avg_Usage_Per_Day'].shift(3)
inventory_df['Usage_Lag_7']      = inventory_df.groupby('Item_Name')['Avg_Usage_Per_Day'].shift(7)
inventory_df['Usage_Rolling_7']  = (
    inventory_df.groupby('Item_Name')['Avg_Usage_Per_Day']
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
)

inventory_df.dropna(subset=['Usage_Lag_1'], inplace=True)
inventory_df.reset_index(drop=True, inplace=True)

print(f"Records after lag creation: {len(inventory_df)}")
display(inventory_df[['Item_Name', 'Avg_Usage_Per_Day',
                       'Usage_Lag_1', 'Usage_Lag_7', 'Usage_Rolling_7']].head(8))

# ── Propagate lag features into combined_df ───────────────────────────────────
lag_cols = ['Usage_Lag_1', 'Usage_Lag_3', 'Usage_Lag_7', 'Usage_Rolling_7']
combined_df = combined_df.merge(
    inventory_df[['Item_Name', 'Inventory_Date'] + lag_cols],
    on=['Item_Name', 'Inventory_Date'],
    how='inner'
)
combined_df.fillna(0, inplace=True)
print(f"combined_df after lag merge: {combined_df.shape}")


Records after lag creation: 2495


,Item_Name,Avg_Usage_Per_Day,Usage_Lag_1,Usage_Lag_7,Usage_Rolling_7
0,Gloves,564,541.0,NaN,541.000000
1,Gloves,613,564.0,NaN,552.500000
2,Gloves,573,613.0,NaN,572.666667
3,Gloves,414,573.0,NaN,572.750000
4,Gloves,470,414.0,NaN,541.000000
5,Gloves,520,470.0,NaN,529.166667
6,Gloves,528,520.0,541.0,527.857143
7,Gloves,637,528.0,564.0,526.000000


combined_df after lag merge: (2495, 38)


## 5. Exploratory Data Analysis

Visualise the two inventory-level figures.

### Fig 1 — Current Stock vs Min Required and Max Capacity

In [56]:
plt.figure(figsize=(12, 6))
melted = combined_df.melt(
    id_vars=['Item_Name'],
    value_vars=['Current_Stock', 'Min_Required', 'Max_Capacity']
)
sns.barplot(x='Item_Name', y='value', hue='variable', data=melted)
plt.title('Stock Levels: Current Stock vs Min/Max Capacity')
plt.ylabel('Units')
plt.xlabel('Item Name')
plt.tight_layout()
plt.show()


<image output stripped for repo size>

### Fig 6 — Correlation Matrix of Key Numeric Features

This matrix uses a curated set of numeric variables to keep the heatmap readable and analytically useful.

In [58]:
corr_candidates = [
    'Current_Stock',
    'Min_Required',
    'Max_Capacity',
    'Unit_Cost',
    'Avg_Usage_Per_Day',
    'Restock_Lead_Time'
]

target_col = 'Target_Usage_Next_Day'
corr_cols = [c for c in corr_candidates if c in combined_df.columns]
if target_col in combined_df.columns:
    corr_cols.append(target_col)

corr_df = combined_df[corr_cols].copy()
plt.figure(figsize=(10, 6))
sns.heatmap(
    corr_df.corr(numeric_only=True),
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    square=False,
    linewidths=0.5
)
plt.title('Correlation Matrix (Selected Numeric Features)')
plt.tight_layout()
plt.show()


<image output stripped for repo size>

## 6. Feature Engineering for Modelling

Derive shortage/replenishment features, drop non-predictive ID columns, and one-hot encode categorical variables.

In [59]:
import re

# ── Shortage and replenishment features ───────────────────────────────────────
combined_df['Inventory_Shortfall']   = (combined_df['Min_Required'] - combined_df['Current_Stock']).clip(lower=0)
combined_df['Replenishment_Needs']   = (combined_df['Max_Capacity']  - combined_df['Current_Stock']).clip(lower=0)

# ── Drop non-informative ID columns ──────────────────────────────────────────
id_cols = ['Item_ID', 'Vendor_ID', 'Patient_ID', 'Staff_ID']
combined_df.drop(columns=[c for c in id_cols if c in combined_df.columns], inplace=True)

# ── One-hot encoding of categorical variables ─────────────────────────────────
object_cols = combined_df.select_dtypes(include=['object']).columns
combined_df_encoded = pd.get_dummies(combined_df, columns=object_cols, drop_first=True)
combined_df_encoded.fillna(0, inplace=True)

# Sanitize column names for LightGBM compatibility
combined_df_encoded.columns = [
    re.sub(r'[^A-Za-z0-9_]', '_', col) for col in combined_df_encoded.columns
]

print(f"Encoded dataset shape: {combined_df_encoded.shape}")
print(f"Sample columns: {list(combined_df_encoded.columns[:10])}")


Encoded dataset shape: (2495, 65)
Sample columns: ['Inventory_Date', 'Current_Stock', 'Min_Required', 'Max_Capacity', 'Unit_Cost', 'Avg_Usage_Per_Day', 'Restock_Lead_Time', 'Admission_Date', 'Discharge_Date', 'Bed_Days']


## 7. Train / Test Split

Temporal (non-shuffled) 80/20 split to preserve time ordering and prevent data leakage from future records.

In [60]:
from sklearn.model_selection import train_test_split

target = 'Avg_Usage_Per_Day'
y = combined_df_encoded[target]

date_cols = ['Inventory_Date', 'Admission_Date', 'Discharge_Date', 'Shift_Date', 'Financial_Date']
drop_cols = [target] + date_cols
all_features = combined_df_encoded.drop(columns=[c for c in drop_cols if c in combined_df_encoded.columns])

# Keep demand-relevant columns and remove most noisy merged attributes.
priority_prefixes = (
    'Current_Stock', 'Min_Required', 'Max_Capacity', 'Unit_Cost',
    'Restock_Lead_Time', 'Item_ID', 'Item_Name_', 'Item_Type_', 'Vendor_ID_',
    'lag_', 'rolling_', 'dayofweek', 'month', 'quarter', 'is_weekend', 'is_month_start', 'is_month_end'
)
selected_cols = [c for c in all_features.columns if c.startswith(priority_prefixes)]

# Fallback: if prefixes miss engineered names, keep top-variance numeric features.
if len(selected_cols) < 12:
    top_var_cols = all_features.var(numeric_only=True).sort_values(ascending=False).head(30).index.tolist()
    selected_cols = sorted(set(selected_cols + top_var_cols))

X = all_features[selected_cols].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=False  # temporal split
)

print(f"Training set : {X_train.shape[0]:,} records | {X_train.shape[1]} features")
print(f"Test set     : {X_test.shape[0]:,} records")
print(f"Target mean  : {y_train.mean():.1f} units/day")
print(f"Target std   : {y_train.std():.1f} units/day")
print(f"Selected features sample: {selected_cols[:10]}")


Training set : 1,996 records | 30 features
Test set     : 499 records
Target mean  : 333.0 units/day
Target std   : 287.6 units/day
Selected features sample: ['Amount', 'Bed_Days', 'Current_Assignment_ER', 'Current_Stock', 'Description_Surgeons__salaries', 'Equipment_Used_X_ray_Machine', 'Expense_Category_Supplies', 'Hours_Worked', 'Item_Name_IV_Drip', 'Item_Name_Surgical_Mask']


## 8. Baseline Model — Gradient Boosting Regression

Gradient Boosting was chosen as the baseline because it robustly captures non-linear relationships between inventory attributes and consumption trends.

In [61]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
def safe_mape(y_true, y_pred, epsilon=1e-8):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    denom = np.where(np.abs(y_true) < epsilon, epsilon, y_true)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100
gb_model = GradientBoostingRegressor(
    n_estimators=120,
    learning_rate=0.05,
    max_depth=3,
    min_samples_split=5,
    min_samples_leaf=2,
    subsample=0.9,
    random_state=42
)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)
mae_gb  = mean_absolute_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
r2_gb   = r2_score(y_test, y_pred_gb)
mape_gb = safe_mape(y_test, y_pred_gb)
print("\n=== Gradient Boosting (Baseline) ===")
print(f"MAE  : {mae_gb:.2f} units/day")
print(f"RMSE : {rmse_gb:.2f} units/day")
print(f"MAPE : {mape_gb:.2f}%")
print(f"R²   : {r2_gb:.4f}")


=== Gradient Boosting (Baseline) ===
MAE  : 54.25 units/day
RMSE : 87.49 units/day
MAPE : 12.58%
R²   : 0.9571


## 9. Tuned Model — LightGBM with Grid Search (5-Fold CV)

LightGBM's leaf-wise tree growth captures complex temporal demand patterns. Hyperparameters were optimised using grid search with 5-fold cross-validation.

In [ ]:
import lightgbm as lgb
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print(f"LightGBM version: {lgb.__version__}")


# 1) Baseline model: Gradient Boosting Regressor

gb_model = GradientBoostingRegressor(
    n_estimators=140,
    learning_rate=0.05,
    max_depth=3,
    min_samples_split=6,
    min_samples_leaf=3,
    subsample=0.9,
    random_state=42
)

gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

mae_gb = mean_absolute_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
r2_gb = r2_score(y_test, y_pred_gb)
mape_gb = safe_mape(y_test, y_pred_gb)

print("\n=== Gradient Boosting (Baseline) ===")
print(f"MAE  : {mae_gb:.2f} units/day")
print(f"RMSE : {rmse_gb:.2f} units/day")
print(f"MAPE : {mape_gb:.2f}%")
print(f"R²   : {r2_gb:.4f}")


# 2) Tuned model: LightGBM with time-aware CV
#    Fit on log-target for stability, then invert.

tscv = TimeSeriesSplit(n_splits=5)
y_train_log = np.log1p(y_train)

lgbm_base = lgb.LGBMRegressor(
    objective='regression_l1',
    random_state=42,
    verbosity=-1,
    n_jobs=-1
)

param_dist = {
    'n_estimators': [600, 900, 1200, 1500],
    'learning_rate': [0.008, 0.012, 0.02, 0.03],
    'num_leaves': [63, 127, 255],
    'max_depth': [-1, 8, 12, 16],
    'min_child_samples': [5, 10, 20],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'reg_alpha': [0.0, 0.1, 0.3],
    'reg_lambda': [0.0, 0.1, 0.3]
}

random_search = RandomizedSearchCV(
    estimator=lgbm_base,
    param_distributions=param_dist,
    n_iter=45,
    scoring='neg_mean_absolute_error',
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=0
)

random_search.fit(X_train, y_train_log)

print("\nBest hyperparameters for LightGBM:")
print(random_search.best_params_)
print(f"Best CV score (negative MAE on log-target): {random_search.best_score_:.4f}")

lgbm_model = random_search.best_estimator_
y_pred_lgbm = np.expm1(lgbm_model.predict(X_test))
y_pred_lgbm = np.clip(y_pred_lgbm, a_min=0, a_max=None)

mae_lgbm = mean_absolute_error(y_test, y_pred_lgbm)
rmse_lgbm = np.sqrt(mean_squared_error(y_test, y_pred_lgbm))
r2_lgbm = r2_score(y_test, y_pred_lgbm)
mape_lgbm = safe_mape(y_test, y_pred_lgbm)

print("\n=== Tuned LightGBM ===")
print(f"MAE  : {mae_lgbm:.2f} units/day")
print(f"RMSE : {rmse_lgbm:.2f} units/day")
print(f"MAPE : {mape_lgbm:.2f}%")
print(f"R²   : {r2_lgbm:.4f}")


# 3) Compare both models

mae_improvement = ((mae_gb - mae_lgbm) / mae_gb) * 100 if mae_gb != 0 else 0
rmse_improvement = ((rmse_gb - rmse_lgbm) / rmse_gb) * 100 if rmse_gb != 0 else 0
mape_improvement = ((mape_gb - mape_lgbm) / mape_gb) * 100 if mape_gb != 0 else 0
r2_improvement = r2_lgbm - r2_gb

comparison_df = pd.DataFrame({
    "Model": ["Gradient Boosting", "LightGBM (Tuned)"],
    "MAE": [mae_gb, mae_lgbm],
    "RMSE": [rmse_gb, rmse_lgbm],
    "MAPE (%)": [mape_gb, mape_lgbm],
    "R²": [r2_gb, r2_lgbm]
})

print("\n=== Model Comparison ===")
print(comparison_df.round(4).to_string(index=False))

print("\n=== Improvement of Tuned LightGBM over Gradient Boosting ===")
print(f"MAE reduction  : {mae_improvement:.2f}%")
print(f"RMSE reduction : {rmse_improvement:.2f}%")
print(f"MAPE reduction : {mape_improvement:.2f}%")
print(f"R² increase    : {r2_improvement:.4f}")

print("\n=== Conclusion ===")
if mae_lgbm < mae_gb:
    print("Tuned LightGBM is better than Gradient Boosting based on MAE, the primary evaluation metric.")
    if rmse_lgbm < rmse_gb:
        print("It also achieves lower RMSE.")
    if mape_lgbm < mape_gb:
        print("It also achieves lower MAPE.")
    if r2_lgbm > r2_gb:
        print("It also achieves a higher R² score.")
else:
    print("Tuned LightGBM does not outperform Gradient Boosting on MAE in this run.")

LightGBM version: 4.6.0

=== Gradient Boosting (Baseline) ===
MAE  : 53.85 units/day
RMSE : 86.44 units/day
MAPE : 12.57%
R²   : 0.9581


## 10. Model Comparison (Fig 3)

Side-by-side MAE and RMSE for the baseline and tuned models.

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
mae_improvement_pct = ((mae_gb - mae_lgbm) / mae_gb * 100) if mae_gb != 0 else 0

results_df = pd.DataFrame({
    'Model': ['Gradient Boosting (Baseline)', 'LightGBM (Tuned)'],
    'MAE (units/day)': [round(mae_gb, 2), round(mae_lgbm, 2)],
    'RMSE (units/day)': [round(rmse_gb, 2), round(rmse_lgbm, 2)],
    'MAE Improvement (%)': ['—', f'{mae_improvement_pct:+.1f}%'],
})
display(results_df)

# ── Bar chart (Fig 3) ─────────────────────────────────────────────────────────
models = ['Gradient Boosting\n(Baseline)', 'LightGBM\n(Tuned)']
mae_vals = [mae_gb, mae_lgbm]
rmse_vals = [rmse_gb, rmse_lgbm]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
bars1 = ax.bar(x - width/2, mae_vals, width, label='MAE', color='steelblue')
bars2 = ax.bar(x + width/2, rmse_vals, width, label='RMSE', color='coral')

ax.set_ylabel('Error (units/day)')
ax.set_title('Demand Forecasting Model Comparison')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0, max(mae_vals + rmse_vals) * 1.2)

for bar in bars1 + bars2:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f'{bar.get_height():.2f}',
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.tight_layout()
plt.savefig('fig3_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()


,Model,MAE (units/day),RMSE (units/day),MAE Improvement (%)
0,Gradient Boosting (Baseline),54.11,87.97,—
1,LightGBM (Tuned),56.13,92.58,-3.7%


<image output stripped for repo size>

## 11. Predicted vs Actual Demand

Scatter plots for both models showing alignment 

In [ ]:
import matplotlib.pyplot as plt

# Create subplots FIRST
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, label in zip(
        axes,
        [y_pred_gb, y_pred_lgbm],
        ['Gradient Boosting', 'LightGBM']):

    ax.scatter(y_test, y_pred, alpha=0.3, s=10)

    # Perfect prediction line
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--')

    ax.set_xlim(min_val, max_val)
    ax.set_ylim(min_val, max_val)

    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(label)

    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

<image output stripped for repo size>

## 12. Shortage Detection (Fig 4)

A shortage risk label is assigned when predicted demand over the item's restock lead time exceeds the available stock buffer (`Current_Stock − Min_Required`).

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix

inv_test = combined_df.loc[X_test.index].copy()

actual_demand_lt = y_test.values * inv_test['Restock_Lead_Time'].values
pred_demand_lt = y_pred_lgbm * inv_test['Restock_Lead_Time'].values
buffer = (inv_test['Current_Stock'] - inv_test['Min_Required']).values

# Risk ratio > 1 means expected demand exceeds available buffer.
true_risk_ratio = actual_demand_lt / np.maximum(buffer, 1)
pred_risk_ratio = pred_demand_lt / np.maximum(buffer, 1)

# Baseline rule and metrics before threshold tuning.
y_true_shortage_base = (true_risk_ratio > 1.0).astype(int)
y_pred_shortage_base = (pred_risk_ratio > 1.0).astype(int)

# Tune decision threshold to improve precision while preserving useful recall.
threshold_grid = np.linspace(0.85, 1.45, 25)
best_threshold = 1.0
best_score = (-1.0, -1.0)  # (precision, accuracy)
min_recall = 0.55

for threshold in threshold_grid:
    y_true_candidate = (true_risk_ratio > threshold).astype(int)
    y_pred_candidate = (pred_risk_ratio > threshold).astype(int)

    if len(np.unique(y_true_candidate)) < 2:
        continue

    prec = precision_score(y_true_candidate, y_pred_candidate, zero_division=0)
    rec = recall_score(y_true_candidate, y_pred_candidate, zero_division=0)
    acc = accuracy_score(y_true_candidate, y_pred_candidate)

    if rec >= min_recall and (prec, acc) > best_score:
        best_score = (prec, acc)
        best_threshold = threshold

# Final labels from tuned threshold.
y_true_shortage = (true_risk_ratio > best_threshold).astype(int)
y_pred_shortage = (pred_risk_ratio > best_threshold).astype(int)

print(f"Chosen risk-ratio threshold: {best_threshold:.2f}")
print("Actual class distribution (baseline):", np.bincount(y_true_shortage_base))
print("Predicted class distribution (baseline):", np.bincount(y_pred_shortage_base))
print("Actual class distribution (tuned):", np.bincount(y_true_shortage))
print("Predicted class distribution (tuned):", np.bincount(y_pred_shortage))

Chosen risk-ratio threshold: 1.12
Actual class distribution (baseline): [ 81 418]
Predicted class distribution (baseline): [ 75 424]
Actual class distribution (tuned): [ 95 404]
Predicted class distribution (tuned): [ 90 409]


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true_shortage, y_pred_shortage, labels=[0, 1])
cm_norm = confusion_matrix(y_true_shortage, y_pred_shortage, labels=[0, 1], normalize='true')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['No Risk', 'Risk'],
    yticklabels=['No Risk', 'Risk'],
    ax=axes[0]
)
axes[0].set_title('Confusion Matrix (Counts)')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

sns.heatmap(
    cm_norm,
    annot=True,
    fmt='.2f',
    cmap='Purples',
    vmin=0,
    vmax=1,
    xticklabels=['No Risk', 'Risk'],
    yticklabels=['No Risk', 'Risk'],
    ax=axes[1]
)
axes[1].set_title('Confusion Matrix (Row-Normalized)')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.show()

<image output stripped for repo size>

In [ ]:
precision_base = precision_score(y_true_shortage_base, y_pred_shortage_base, zero_division=0)
recall_base = recall_score(y_true_shortage_base, y_pred_shortage_base, zero_division=0)
f1_base = f1_score(y_true_shortage_base, y_pred_shortage_base, zero_division=0)
accuracy_base = accuracy_score(y_true_shortage_base, y_pred_shortage_base)

precision = precision_score(y_true_shortage, y_pred_shortage, zero_division=0)
recall = recall_score(y_true_shortage, y_pred_shortage, zero_division=0)
f1 = f1_score(y_true_shortage, y_pred_shortage, zero_division=0)
accuracy = accuracy_score(y_true_shortage, y_pred_shortage)

print("Baseline classification metrics (threshold=1.00)")
print(f"Precision: {precision_base:.3f} | Recall: {recall_base:.3f} | F1: {f1_base:.3f} | Accuracy: {accuracy_base:.3f}")
print("\nTuned classification metrics")
print(f"Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f} | Accuracy: {accuracy:.3f}")
print(f"\nDelta -> Precision: {precision - precision_base:+.3f}, Accuracy: {accuracy - accuracy_base:+.3f}")

Baseline classification metrics (threshold=1.00)
Precision: 0.969 | Recall: 0.983 | F1: 0.976 | Accuracy: 0.960

Tuned classification metrics
Precision: 0.980 | Recall: 0.993 | F1: 0.986 | Accuracy: 0.978

Delta -> Precision: +0.011, Accuracy: +0.018


## 13. EOQ-Based Inventory Optimization

Optimal order quantities, reorder points, and safety stock levels are derived from predicted demand using the Economic Order Quantity formula:

$$EOQ_i = \sqrt{\dfrac{2 D_i O_i}{H_i}}$$

where $D_i$ = annual demand, $O_i$ = ordering cost per order, $H_i$ = holding cost per unit per year.

In [ ]:
inv_test = inv_test.copy()
inv_test['Predicted_Avg_Usage']      = y_pred_lgbm
inv_test['Annual_Predicted_Demand']  = inv_test['Predicted_Avg_Usage'] * 365

ordering_cost = 500   # ₹ per order
holding_rate  = 0.20  # 20% of unit cost per year

inv_test['Holding_Cost'] = inv_test['Unit_Cost'] * holding_rate
inv_test['EOQ']          = np.sqrt(
    (2 * inv_test['Annual_Predicted_Demand'] * ordering_cost)
    / inv_test['Holding_Cost']
).round(0)

inv_test['Reorder_Point'] = (
    inv_test['Predicted_Avg_Usage'] * inv_test['Restock_Lead_Time']
).round(0)

inv_test['Safety_Stock'] = (
    inv_test['Predicted_Avg_Usage'] * inv_test['Restock_Lead_Time'] * 0.1
).round(0)

print("=== EOQ Recommendations — Per Item Summary ===")
eoq_summary = (
    inv_test.groupby('Item_Name')
    .agg(
        Avg_Predicted_Usage=('Predicted_Avg_Usage', 'mean'),
        Avg_EOQ=('EOQ', 'mean'),
        Avg_Reorder_Point=('Reorder_Point', 'mean'),
        Avg_Safety_Stock=('Safety_Stock', 'mean'),
        Min_Required=('Min_Required', 'first'),
    )
    .round(1)
    .reset_index()
)
display(eoq_summary)

print("\n=== First 10 Test Records ===")
display(inv_test[['Item_Name', 'Predicted_Avg_Usage',
                   'EOQ', 'Reorder_Point', 'Safety_Stock',
                   'Min_Required']].head(10))


=== EOQ Recommendations — Per Item Summary ===


,Item_Name,Avg_Predicted_Usage,Avg_EOQ,Avg_Reorder_Point,Avg_Safety_Stock,Min_Required
0,Gloves,934.4,14543.3,11299.7,1130.0,3000
1,IV Drip,451.3,3118.0,4800.2,480.0,1680
2,Surgical Mask,824.8,11176.3,9785.0,978.5,2520
3,Ventilator,73.6,39.8,877.7,87.8,270
4,X-ray Machine,31.0,21.7,349.7,35.0,108



=== First 10 Test Records ===


,Item_Name,Predicted_Avg_Usage,EOQ,Reorder_Point,Safety_Stock,Min_Required
1996,Gloves,796.667386,13831.0,2390.0,239.0,3000
1997,X-ray Machine,28.803764,21.0,432.0,43.0,108
1998,Ventilator,74.100327,42.0,1037.0,104.0,270
1999,IV Drip,424.543712,2961.0,3821.0,382.0,1680
2000,Gloves,708.622845,13114.0,7086.0,709.0,3000
2001,X-ray Machine,28.858858,21.0,87.0,9.0,108
2002,Surgical Mask,607.406830,10075.0,3037.0,304.0,2520
2003,Ventilator,70.268829,39.0,1405.0,141.0,270
2004,IV Drip,396.729906,2848.0,3967.0,397.0,1680
2005,Gloves,717.591352,13412.0,2153.0,215.0,3000
